# Sunny–shaded timing difference across continents

The runoff-onset timing difference between sun-facing (CHILI warm) and shaded (CHILI cool)
pixels as a function of the 10-year median onset and latitude (the seasonal-modulation figure),
plus the MAD-vs-onset view. Same continents cube and thresholds as `lat_elev_binning.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from gsro_analysis import aggregate, paths, settings

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

PIX_THRESH = 1000   # a (continent, latitude, elevation) bin needs at least 1000 pixels to be shown

# the continents cube written by 0_aggregate_by_continent.ipynb: continent x latitude x elevation x chili_class x
# water_year, with <var> = bin mean, <var>_std, <var>_n = pixel count (gsro_analysis.aggregate); fcf_lte_50 = the
# analyses' pixel filter
continents_path = paths.aggregation_dir('continents', config.version) / 'all_continents_fcf_lte_50.nc'
continents_cube_ds = xr.open_dataset(continents_path)
continents_cube_ds

In [ ]:
# all insolation classes together (count-weighted), thresholded
continents_ds = aggregate.threshold(aggregate.collapse(continents_cube_ds), PIX_THRESH)

# sunny (warm) minus shaded (cool) CHILI classes, each thresholded on its own count
cool_ds = aggregate.threshold(continents_cube_ds.sel(chili_class='cool'), PIX_THRESH)
warm_ds = aggregate.threshold(continents_cube_ds.sel(chili_class='warm'), PIX_THRESH)
continents_ds['chili_warm_cool_difference'] = warm_ds['runoff_onset_median'] - cool_ds['runoff_onset_median']
continents_ds['chili_warm_cool_ratio'] = warm_ds['runoff_onset_median'] / cool_ds['runoff_onset_median']
continents_ds['chili_warm_cool_n'] = xr.ufuncs.minimum(warm_ds['runoff_onset_median_n'], cool_ds['runoff_onset_median_n'])

# GTOPO30 land-pixel histogram: the grey background of every panel (all land, not just mapped pixels)
dem_pixel_count_da = continents_cube_ds['dem_pixel_count']
continents_ds

In [ ]:
continent_order = ['North America', 'Europe', 'Asia', 'South America', 'Africa', 'Oceania']
continents_ds = continents_ds.reindex({'continent': continent_order})
continents_ds

## Timing offset vs median onset

In [ ]:
f,axs=plt.subplots(1,2,figsize=(20,10))
continents_ds.plot.scatter(ax=axs[0],x='runoff_onset_median', y='chili_warm_cool_difference',hue='latitude',cmap='RdYlGn')
continents_ds.plot.scatter(ax=axs[1],x='runoff_onset_median', y='chili_warm_cool_difference',hue='elevation')

In [ ]:
scatterplot = continents_ds.reindex({'continent': continent_order}).plot.scatter(col='continent',
                                                           col_wrap=3,
                                                           x='runoff_onset_median', 
                                                           y='chili_warm_cool_difference',
                                                           #hue='elevation',
                                                           hue='latitude',
                                                           s=10, 
                                                        #    vmin=30,
                                                        #    vmax=70,
                                                           cmap='Spectral',
                                                        #    cmap='jet',
                                                           #vmax=6000
                                                           )
#for ax in scatterplot

In [ ]:
f,axs=plt.subplots(nrows=2,ncols=3,figsize=(10,6),sharex=True,sharey=True,dpi=300, layout='constrained')


cmap='rainbow'
cmap='gnuplot'
cmap='turbo'
cmap='rainbow'
# cmap = plt.cm.gnuplot

# start = 0.1
# end = 0.9
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
#     'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=start, b=end),
#     cmap(np.linspace(start, end, 256))
# )
import matplotlib.ticker as mticker

# Define a custom formatter for latitude ticks
def format_latitude(value, pos):
    if value > 0:
        return f"{int(value)}°N"
    elif value < 0:
        return f"{int(abs(value))}°S"
    else:
        return "0°"

bottom_row_lat_min = -55
bottom_row_lat_max = -30

top_row_lat_min = 25
top_row_lat_max = 70

for i, continent in enumerate(continent_order):

    if continent == 'Asia' or continent == 'Oceania':
        colorbar = True
    else:
        colorbar = False

    if i < 3:  # Top row (Northern Hemisphere)
        scatter=continents_ds.sel(continent=continent).plot.scatter(ax=axs[0,i], 
                                                                              x='runoff_onset_median', 
                                                                              y='chili_warm_cool_difference', 
                                                                              hue='latitude', 
                                                                              cmap=f'{cmap}_r', 
                                                                              vmin=top_row_lat_min, 
                                                                              vmax=top_row_lat_max, 
                                                                              #edgecolor='black',
                                                                              #linewidth=0.1,
                                                                              s=15,
                                                                              add_colorbar=colorbar,
                                                                              cbar_kwargs={'label': 'Latitude [degrees]'}
                                                                              )
        axs[0,i].set_title(continent)
        axs[0,i].set_xlabel('')
        #axs[0,i].set_ylim([top_row_lat_min, top_row_lat_max])
    else:  # Bottom row (Southern Hemisphere)
        scatter=continents_ds.sel(continent=continent).plot.scatter(ax=axs[1,i-3], 
                                                                              x='runoff_onset_median', 
                                                                              y='chili_warm_cool_difference', 
                                                                              hue='latitude', 
                                                                              cmap=cmap, 
                                                                              vmin=bottom_row_lat_min, 
                                                                              vmax=bottom_row_lat_max, 
                                                                              
                                                                              #edgecolor='black',
                                                                              #linewidth=0.0001,
                                                                              s=15,
                                                                              add_colorbar=colorbar,
                                                                              cbar_kwargs={'label': 'Latitude [degrees]'},
                                                                              )                                                                             
        axs[1,i-3].set_title(continent)
        #axs[1,i-3].set_ylim([bottom_row_lat_min, bottom_row_lat_max])
        axs[1,i-3].set_xlabel('10-year median runoff onset [DOWY]')
        axs[1,i-3].set_xlabel('')


    if colorbar:
        scatter.colorbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_latitude))

    
for ax in axs.flatten():
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_ylim([-60, 20])  # Full latitude range
    ax.set_xlim([110, 310])
    ax.grid(True, linestyle='--', alpha=0.5)
    if ax in axs[:,0]:
        ax.set_ylabel('Time offset between\nhigh/low solar areas\n[days]')
        ax.set_ylabel('')
    else:
        ax.set_ylabel('')

# make just one x and y label for the whole figure
f.text(0.5, -0.03, '10-year median runoff onset [DOWY]', ha='center', fontsize=12)
f.text(-0.02, 0.5, 'Time offset between high/low solar areas [days]', va='center', rotation='vertical', fontsize=12)

## Seasonal modulation: median onset vs latitude, coloured by sunny–shaded offset

In [ ]:
f, axs = plt.subplots(nrows=2, ncols=3, figsize=(10, 6), sharex='row', sharey='row', dpi=300, layout='constrained')

import matplotlib.ticker as mticker
from matplotlib.dates import MonthLocator, DateFormatter
import datetime

# Define a custom formatter for latitude ticks
def format_latitude(value, pos):
    if value > 0:
        return f"{int(value)}°N"
    elif value < 0:
        return f"{int(abs(value))}°S"
    else:
        return "0°"

# Function to convert DOWY to month name for Northern Hemisphere
def dowy_to_month_nh(dowy):
    """Convert DOWY to month for Northern Hemisphere (Oct 1 = DOWY 1)"""
    base_date = datetime.date(2020, 10, 1)  # Using 2020 as leap year
    date = base_date + datetime.timedelta(days=int(dowy) - 1)
    return date.strftime('%b')

# Function to convert DOWY to month name for Southern Hemisphere
def dowy_to_month_sh(dowy):
    """Convert DOWY to month for Southern Hemisphere (Apr 1 = DOWY 1)"""
    base_date = datetime.date(2020, 4, 1)  # Using 2020 as leap year
    date = base_date + datetime.timedelta(days=int(dowy) - 1)
    return date.strftime('%b')

# Calculate month ticks dynamically
def get_month_ticks(hemisphere='northern'):
    """Get DOWY values for the first of each month"""
    if hemisphere == 'northern':
        # Start from Oct 1
        base_date = datetime.date(2020, 10, 1)
        start_month = 10
        months_in_order = list(range(10, 13)) + list(range(1, 10))  # Oct-Dec, Jan-Sep
    else:
        # Start from Apr 1
        base_date = datetime.date(2020, 4, 1)
        start_month = 4
        months_in_order = list(range(4, 13)) + list(range(1, 4))  # Apr-Dec, Jan-Mar
    
    ticks = []
    current_date = base_date
    ticks.append(1)  # First day is always DOWY 1
    
    # For each month in the water year (11 more months after the first)
    for i in range(1, 12):
        # Move to first day of next month
        if current_date.month == 12:
            current_date = datetime.date(current_date.year + 1, 1, 1)
        else:
            current_date = datetime.date(current_date.year, current_date.month + 1, 1)
        
        # Calculate DOWY
        dowy = (current_date - base_date).days + 1
        ticks.append(dowy)
    
    return ticks

# Get month ticks for both hemispheres
nh_month_ticks = get_month_ticks('northern')
sh_month_ticks = get_month_ticks('southern')

# Get month labels
nh_month_labels = [dowy_to_month_nh(d) for d in nh_month_ticks]
sh_month_labels = [dowy_to_month_sh(d) for d in sh_month_ticks]

# Define latitude range for y-axis
lat_min = -60
lat_max = 70

# Color scale range for time offset
offset_min = -30
offset_max = 10

for i, continent in enumerate(continent_order):

    if continent == 'Asia' or continent == 'Oceania':
        colorbar = True
    else:
        colorbar = False

    # Get data for this continent
    cont_data = continents_ds.sel(continent=continent)
    
    # Calculate DOWY of max time offset for each latitude (with percentile approach)
    max_offset_dowy = []
    lats = []
    percentile_threshold = 10  # Lower 10th percentile for strongest offsets
    
    for lat in cont_data.latitude.values:
        lat_data = cont_data.sel(latitude=lat)
        if not lat_data['chili_warm_cool_difference'].isnull().all():
            # Find points where time offset is in the lower percentile
            offset_vals = lat_data['chili_warm_cool_difference'].values
            threshold = np.nanpercentile(offset_vals, percentile_threshold)
            mask = offset_vals <= threshold
            
            # Weighted average of DOWY for these strong-offset points
            if np.any(mask):
                dowy_vals = lat_data['runoff_onset_median'].values[mask]
                weights = np.abs(offset_vals[mask])  # Weight by magnitude of offset
                avg_dowy = np.average(dowy_vals, weights=weights)
                max_offset_dowy.append(float(avg_dowy))
                lats.append(lat)
    
    if i < 3:  # Top row (Northern Hemisphere)
        scatter = cont_data.plot.scatter(
            ax=axs[0, i], 
            x='runoff_onset_median', 
            y='latitude', 
            hue='chili_warm_cool_difference', 
            cmap='plasma', 
            vmin=offset_min, 
            vmax=offset_max, 
            s=15,
            add_colorbar=colorbar,
            alpha=1.0,
            cbar_kwargs={'label': ''}
        )
        axs[0, i].set_title(continent)
        axs[0, i].set_xlabel('')
        axs[0, i].set_ylim([20, 72])
        
        # Plot the red line
        # axs[0, i].plot(max_offset_dowy, lats, 'r-', linewidth=2, marker='o', 
        #                markersize=4, label='Max time offset', zorder=10)
        
        # Set month ticks for Northern Hemisphere
        axs[0, i].set_xticks(nh_month_ticks)
        axs[0, i].set_xticklabels(nh_month_labels, rotation=45, ha='right')
        
    else:  # Bottom row (Southern Hemisphere)
        scatter = cont_data.plot.scatter(
            ax=axs[1, i-3], 
            x='runoff_onset_median', 
            y='latitude', 
            hue='chili_warm_cool_difference', 
            cmap='plasma', 
            vmin=offset_min, 
            vmax=offset_max, 
            s=15,
            add_colorbar=colorbar,
            alpha=1.0,
            cbar_kwargs={'label': ''}
        )
        axs[1, i-3].set_title(continent)
        axs[1, i-3].set_xlabel('')
        axs[1, i-3].set_ylim([-60, -28])
        
        # Plot the red line
        # axs[1, i-3].plot(max_offset_dowy, lats, 'r-', linewidth=2, marker='o', 
        #                  markersize=4, label='Max time offset', zorder=10)
        
        # Set month ticks for Southern Hemisphere
        axs[1, i-3].set_xticks(sh_month_ticks)
        axs[1, i-3].set_xticklabels(sh_month_labels, rotation=45, ha='right')

        
for ax in axs.flatten():
    ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_xlim([110, 310])
    ax.grid(True, linestyle='--', alpha=0.5)
    
    # Format y-axis with latitude labels
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_latitude))
    
    if ax in axs[:, 0]:
        ax.set_ylabel('')
    else:
        ax.set_ylabel('')

axs[1,1].set_visible(False)

# Make just one x and y label for the whole figure
f.text(0.5, -0.03, '10-year median runoff onset', ha='center', fontsize=12)
f.text(-0.02, 0.5, 'Latitude [degrees]', va='center', rotation='vertical', fontsize=12)
f.text(1.0, 0.5, 'Runoff onset timing difference between sunny and shaded areas [days]', 
       va='center', fontsize=12, rotation='vertical')
f.savefig(paths.figdir('continents', config.version) / 'global_scatter_latitude_vs_onset.png', dpi=300, bbox_inches='tight')

## MAD vs median onset

In [ ]:
f,axs=plt.subplots(nrows=2,ncols=3,figsize=(10,6),sharex=True,sharey=True,dpi=300, layout='constrained')


cmap='rainbow'
cmap='gnuplot'
cmap='turbo'
cmap='rainbow'
# cmap = plt.cm.gnuplot

# start = 0.1
# end = 0.9
# cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
#     'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=start, b=end),
#     cmap(np.linspace(start, end, 256))
# )
import matplotlib.ticker as mticker

# Define a custom formatter for latitude ticks
def format_latitude(value, pos):
    if value > 0:
        return f"{int(value)}°N"
    elif value < 0:
        return f"{int(abs(value))}°S"
    else:
        return "0°"

bottom_row_lat_min = -55
bottom_row_lat_max = -30

top_row_lat_min = 25
top_row_lat_max = 70

for i, continent in enumerate(continent_order):

    if continent == 'Asia' or continent == 'Oceania':
        colorbar = True
    else:
        colorbar = False

    if i < 3:  # Top row (Northern Hemisphere)
        scatter=continents_ds.sel(continent=continent).plot.scatter(ax=axs[0,i], 
                                                                              x='runoff_onset_median', 
                                                                              y='runoff_onset_mad', 
                                                                              hue='latitude', 
                                                                              cmap=f'{cmap}_r',
                                                                              vmin=top_row_lat_min, 
                                                                              vmax=top_row_lat_max, 
                                                                              #edgecolor='black',
                                                                              #linewidth=0.1,
                                                                              s=15,
                                                                              add_colorbar=colorbar,
                                                                              cbar_kwargs={'label': 'Latitude [degrees]'}
                                                                              )
        axs[0,i].set_title(continent)
        axs[0,i].set_xlabel('')
        #axs[0,i].set_ylim([top_row_lat_min, top_row_lat_max])
    else:  # Bottom row (Southern Hemisphere)
        scatter=continents_ds.sel(continent=continent).plot.scatter(ax=axs[1,i-3], 
                                                                              x='runoff_onset_median', 
                                                                              y='runoff_onset_mad', 
                                                                              hue='latitude', 
                                                                              cmap=cmap, 
                                                                              vmin=bottom_row_lat_min, 
                                                                              vmax=bottom_row_lat_max, 
                                                                              
                                                                              #edgecolor='black',
                                                                              #linewidth=0.0001,
                                                                              s=15,
                                                                              add_colorbar=colorbar,
                                                                              cbar_kwargs={'label': 'Latitude [degrees]'},
                                                                              )                                                                             
        axs[1,i-3].set_title(continent)
        #axs[1,i-3].set_ylim([bottom_row_lat_min, bottom_row_lat_max])
        axs[1,i-3].set_xlabel('10-year median runoff onset [DOWY]')
        axs[1,i-3].set_xlabel('')


    if colorbar:
        scatter.colorbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(format_latitude))

    
for ax in axs.flatten():
    #ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
    ax.set_ylim([0, 60])  # Full latitude range
    ax.set_xlim([110, 310])
    #ax.grid(True, linestyle='--', alpha=0.5)
    if ax in axs[:,0]:
        ax.set_ylabel('10-year runoff onset median absolute deviation [days]')
        ax.set_ylabel('')
    else:
        ax.set_ylabel('')

# make just one x and y label for the whole figure
f.text(0.5, -0.03, '10-year median runoff onset [DOWY]', ha='center', fontsize=12)
f.text(-0.02, 0.5, '10-year runoff onset median absolute deviation [days]', va='center', rotation='vertical', fontsize=12)